In [1]:
import glob
import json
import math

import folium
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

# Mirrored from backend/src/config/bounds.ts — keep in sync, do not invent values.
BOUNDS = {"min_lat": -7.615, "max_lat": -7.6, "min_lng": 110.195, "max_lng": 110.215}
CENTER = ((BOUNDS["min_lat"] + BOUNDS["max_lat"]) / 2, (BOUNDS["min_lng"] + BOUNDS["max_lng"]) / 2)
EARTH_RADIUS_M = 6_371_000

SAMPLE_GLOB = "../sample/*.json"


## 1. Load samples



In [2]:
frames = []
for path in sorted(glob.glob(SAMPLE_GLOB)):
    with open(path) as f:
        payload = json.load(f)
    frame = pd.DataFrame(payload["data"])
    frame["file"] = path.split("/")[-1]
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)
df["_updated_at"] = pd.to_datetime(df["_updated_at"], format="ISO8601")  # real data mixes ms and no-ms precision
df["device"] = df["client_id"].str[-4:]  # short label for plots

print(f"{len(df)} rows, {df['client_id'].nunique()} devices")
df.groupby("device").agg(
    points=("latitude", "size"),
    unique_coords=("latitude", "nunique"),
    first=("_updated_at", "min"),
    last=("_updated_at", "max"),
)


1619 rows, 2 devices


,points,unique_coords,first,last
device,,,,
ce74,174,106,2026-07-08 00:40:58.994000+00:00,2026-07-08 09:01:07.754000+00:00
dc6a,1445,261,2026-06-06 16:43:44.083000+00:00,2026-07-08 02:29:11.112000+00:00


## 2. Regional overview map

In [3]:
DEVICE_COLORS = ["#2563eb", "#d97706", "#059669", "#dc2626", "#7c3aed"]
device_color = {d: DEVICE_COLORS[i % len(DEVICE_COLORS)] for i, d in enumerate(sorted(df["device"].unique()))}

overview = folium.Map(location=CENTER, zoom_start=11, tiles="OpenStreetMap")
folium.Rectangle(
    bounds=[(BOUNDS["min_lat"], BOUNDS["min_lng"]), (BOUNDS["max_lat"], BOUNDS["max_lng"])],
    color="red", weight=2, fill=False, tooltip="BOROBUDUR_BOUNDS",
).add_to(overview)

for row in df.itertuples():
    folium.CircleMarker(
        location=(row.latitude, row.longitude),
        radius=2, weight=0,
        fill=True, fill_opacity=0.6, fill_color=device_color[row.device],
    ).add_to(overview)

overview


## 3. Bounds filter


In [4]:
in_bounds = df[
    df["latitude"].between(BOUNDS["min_lat"], BOUNDS["max_lat"])
    & df["longitude"].between(BOUNDS["min_lng"], BOUNDS["max_lng"])
].copy()

print(f"in bounds: {len(in_bounds)} / {len(df)} rows "
      f"({len(in_bounds[['latitude', 'longitude']].drop_duplicates())} unique coordinates)")
in_bounds.groupby("device").size().rename("points_in_bounds").to_frame()


in bounds: 81 / 1619 rows (65 unique coordinates)


,points_in_bounds
device,
ce74,44
dc6a,37


## 4. DBSCAN


In [ ]:
def run_dbscan(points: pd.DataFrame, eps_m: float, min_samples: int) -> pd.DataFrame:
    """Cluster lat/lng points; returns a copy with a `cluster` column (-1 = noise)."""
    out = points.copy()
    coords_rad = np.radians(out[["latitude", "longitude"]].to_numpy())
    labels = DBSCAN(
        eps=eps_m / EARTH_RADIUS_M, min_samples=min_samples, metric="haversine"
    ).fit_predict(coords_rad)
    out["cluster"] = labels
    return out

EPS_M = 8
MIN_SAMPLES = 5

clustered = run_dbscan(in_bounds, EPS_M, MIN_SAMPLES)
summary = (
    clustered.groupby("cluster")
    .agg(points=("latitude", "size"), center_lat=("latitude", "mean"), center_lng=("longitude", "mean"))
    .sort_values("points", ascending=False)
)
print(f"eps={EPS_M}m min_samples={MIN_SAMPLES}: "
      f"{(summary.index != -1).sum()} clusters, "
      f"{int(summary.loc[-1, 'points']) if -1 in summary.index else 0} noise points")
summary


eps=8m min_samples=5: 4 clusters, 35 noise points


,points,center_lat,center_lng
cluster,,,
-1,35,-7.608586,110.206354
0,22,-7.607974,110.204051
1,9,-7.607943,110.204246
3,8,-7.608089,110.203796
2,7,-7.610299,110.208096


## 5. Cluster map


In [6]:
CLUSTER_COLORS = ["#2563eb", "#d97706", "#059669", "#dc2626", "#7c3aed",
                  "#0891b2", "#ca8a04", "#be185d", "#4d7c0f", "#6d28d9"]

cluster_map = folium.Map(location=CENTER, zoom_start=16, tiles="OpenStreetMap")
folium.Rectangle(
    bounds=[(BOUNDS["min_lat"], BOUNDS["min_lng"]), (BOUNDS["max_lat"], BOUNDS["max_lng"])],
    color="red", weight=1, fill=False,
).add_to(cluster_map)

for row in clustered.itertuples():
    is_noise = row.cluster == -1
    folium.CircleMarker(
        location=(row.latitude, row.longitude),
        radius=3 if not is_noise else 2, weight=0,
        fill=True,
        fill_opacity=0.8 if not is_noise else 0.35,
        fill_color="#9ca3af" if is_noise else CLUSTER_COLORS[row.cluster % len(CLUSTER_COLORS)],
    ).add_to(cluster_map)

for cluster_id, row in summary.iterrows():
    if cluster_id == -1:
        continue
    folium.Marker(
        location=(row["center_lat"], row["center_lng"]),
        popup=f"cluster {cluster_id}: {int(row['points'])} points",
        icon=folium.Icon(color="black", icon="fire", prefix="fa"),
    ).add_to(cluster_map)

cluster_map


## 6. Parameter sweep

Cluster/noise counts across a grid of `eps` / `min_samples`. Look for a plateau —
a region where small parameter changes stop changing the answer.


In [7]:
rows = []
for eps_m in [10, 20, 30, 50, 75, 100]:
    for min_samples in [3, 5, 10]:
        labels = run_dbscan(in_bounds, eps_m, min_samples)["cluster"]
        rows.append({
            "eps_m": eps_m,
            "min_samples": min_samples,
            "clusters": labels[labels != -1].nunique(),
            "noise_points": int((labels == -1).sum()),
            "clustered_points": int((labels != -1).sum()),
        })
pd.DataFrame(rows).pivot(index="eps_m", columns="min_samples",
                         values=["clusters", "noise_points"])


clusters       noise_points        
min_samples       3  5  10           3   5   10
eps_m                                          
10                 3  3  1           34  34  50
20                 3  4  1           29  29  48
30                 4  2  1           23  29  37
50                 5  2  1           12  25  35
75                 2  3  2            6   7  20
100                1  1  2            0   3  18

## 7. Export preview in `hotspots.json` shape


In [8]:
hotspots = [
    {
        "cluster_id": f"cluster_{cluster_id}",
        "center_lat": round(row["center_lat"], 7),
        "center_lng": round(row["center_lng"], 7),
        "total_points": int(row["points"]),
        "label": f"Hotspot {cluster_id + 1}",
    }
    for cluster_id, row in summary.iterrows()
    if cluster_id != -1
]

out_path = "../output/hotspots_preview.json"
with open(out_path, "w") as f:
    json.dump({"hotspots": hotspots, "params": {"eps_m": EPS_M, "min_samples": MIN_SAMPLES}}, f, indent=2)

print(f"wrote {len(hotspots)} hotspots -> {out_path}")
hotspots


wrote 4 hotspots -> ../output/hotspots_preview.json


[{'cluster_id': 'cluster_0',
  'center_lat': np.float64(-7.6079744),
  'center_lng': np.float64(110.2040508),
  'total_points': 22,
  'label': 'Hotspot 1'},
 {'cluster_id': 'cluster_1',
  'center_lat': np.float64(-7.6079434),
  'center_lng': np.float64(110.2042457),
  'total_points': 9,
  'label': 'Hotspot 2'},
 {'cluster_id': 'cluster_3',
  'center_lat': np.float64(-7.6080895),
  'center_lng': np.float64(110.203796),
  'total_points': 8,
  'label': 'Hotspot 4'},
 {'cluster_id': 'cluster_2',
  'center_lat': np.float64(-7.6102991),
  'center_lng': np.float64(110.2080957),
  'total_points': 7,
  'label': 'Hotspot 3'}]